# BDD100K-pretrained ResNet-50 scene embedding

Standalone experiment, not wired into `feature_pipeline_v3_3class.ipynb` -- see this
folder's `README.md` for the hypothesis, source model details, and caveats (in particular:
this was authored in an environment with no `torch`, no local feature data, and no network
route to the checkpoint host, so run it end-to-end here before trusting the output).

Follows the same template as the DINOv2 embedding cells in
`notebooks/feature_pipeline_v3_3class.ipynb`: load a pretrained backbone, strip the
classification head, run the same `CENTER_SEG` crop of each sequence's target frame through
it, save the penultimate-layer embedding as a `.npy`, benchmark against the existing feature
sets.

Backbone: **ResNet-50**, trained on **BDD100K scene tagging** (tunnel / residential / parking
lot / city street / gas station / highway), from the
[SysCV/bdd100k-models](https://github.com/SysCV/bdd100k-models) model zoo (`tagging/`
task, `resnet50_5x_224x224_scene_tag_bdd100k` config -- val accuracy 77.66%).

## 0. Setup -- paths, manifest, existing base features

Mirrors the setup in `feature_pipeline_v3_3class.ipynb` cells 2/4/6, read-only: this reloads and re-filters the same base arrays so the benchmark cell below can compare against them, without modifying that notebook or its outputs.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import hashlib
import json
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

from standard_e2e import Modality


def find_repo_root(start=None, marker='data/train_manifest.json'):
    """Walk upward from `start` until `marker` is found. Same idea as
    scripts/bench_common.py's resolve_paths, generalized to work regardless of whether
    this notebook is opened from the repo root or from this folder."""
    cur = Path(start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'Could not find {marker!r} above {cur}')


ROOT          = find_repo_root()
TRAIN_DIR     = ROOT / 'data' / 'processed' / 'waymo_e2e' / 'training'
MANIFEST_PATH = ROOT / 'data' / 'train_manifest.json'
FEATURES_DIR  = ROOT / 'data' / 'processed' / 'waymo_e2e' / 'features'
CKPT_DIR      = Path.cwd() / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES      = ['straight', 'left-turn', 'right-turn']
RANDOM_STATE = 42
CENTER_SEG   = (128, 256)  # front-center camera segment, same as feature_pipeline_v3_3class.ipynb

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

print(f'ROOT:          {ROOT}')
print(f'FEATURES_DIR:  {FEATURES_DIR}')
print(f'Loaded {len(manifest)} sequences from manifest')


def load_frame(fname):
    data = np.load(TRAIN_DIR / fname, allow_pickle=True)
    return np.array(data['_modality_data'].item()[Modality.CAMERAS])


ROOT:          C:\Users\hi2ni\281-s2-group2-final
FEATURES_DIR:  C:\Users\hi2ni\281-s2-group2-final\data\processed\waymo_e2e\features
Loaded 2037 sequences from manifest


In [2]:
BASE_FILES = {
    'hog': 'hog.npy', 'hsv': 'hsv.npy', 'yolo': 'yolo.npy', 'seq_ids': 'seq_ids.npy',
}
missing = [f for f in BASE_FILES.values() if not (FEATURES_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        'Missing base feature files: ' + ', '.join(missing) +
        '\nRun waymo_feature_extraction.ipynb (or pull features from the team drive) first.'
    )

base = {
    name: np.load(FEATURES_DIR / filename, allow_pickle=name == 'seq_ids')
    for name, filename in BASE_FILES.items()
}
base['seq_ids'] = base['seq_ids'].astype(str)
# labels.npy isn't in FEATURES_DIR -- derive labels from the manifest instead. Confirmed
# seq_ids.npy[class_mask] == seq_ids_3class.npy exactly, so this lines up row-for-row with
# every other _3class array below. (labels_CNN_2D.npy / labels_CNN_3D.npy are a different,
# 1966-row split used only by the CNN notebooks -- wrong row count/order for this one.)
base['labels'] = np.array([manifest[sid]['label'] for sid in base['seq_ids']])

class_mask  = np.isin(base['labels'], CLASSES)
labels_3c   = base['labels'][class_mask]
seq_ids_3c  = base['seq_ids'][class_mask]
hog_3c      = base['hog'][class_mask]
hsv_3c      = base['hsv'][class_mask]
yolo_3c     = base['yolo'][class_mask]

majority_baseline_3c = pd.Series(labels_3c).value_counts(normalize=True).max()
print(f'3-class subset: {len(labels_3c)} rows, majority baseline {majority_baseline_3c:.3f}')
print(Counter(labels_3c))

# Optional extras: already filtered to the 3-class subset on disk (row-for-row aligned with
# seq_ids_3c above), loaded as-is if present -- used for combined feature-set comparisons in
# the benchmark cell, skipped if absent.
road_v2_3c = None
if (FEATURES_DIR / 'road_v2_3class.npy').exists():
    road_v2_3c = np.load(FEATURES_DIR / 'road_v2_3class.npy')
    print(f'road_v2_3c: {road_v2_3c.shape}')

dino_3c = None
if (FEATURES_DIR / 'dino_v2_3class.npy').exists():
    dino_3c = np.load(FEATURES_DIR / 'dino_v2_3class.npy')
    print(f'dino_3c:    {dino_3c.shape}')


3-class subset: 1604 rows, majority baseline 0.464
Counter({'straight': 745, 'right-turn': 467, 'left-turn': 392})
road_v2_3c: (1604, 10)
dino_3c:    (1604, 384)


## 1. Download the BDD100K ResNet-50 scene-tagging checkpoint

Source: [SysCV/bdd100k-models](https://github.com/SysCV/bdd100k-models),
`tagging/configs/scene/resnet50_5x_224x224_scene_tag_bdd100k.py`. Cached locally under
`checkpoints/` so re-running the notebook doesn't re-download.

**The model zoo's own host (`dl.cv.ethz.ch`) is currently down** -- confirmed via DNS
lookups against multiple public resolvers (all return NXDOMAIN) and a matching, still-open
upstream report: [SysCV/bdd100k-models#28](https://github.com/SysCV/bdd100k-models/issues/28)
("dl.cv.ethz.ch domain is down"), filed 2026-03-25, unresolved as of this notebook being
written (2026-07-31). This is not specific to this machine/environment.

**Working fallback used here:** the Internet Archive's Wayback Machine crawled this exact
file on 2025-01-29 (before the host died) — `http://web.archive.org/web/20250129052906if_/https://dl.cv.ethz.ch/...pth`.
This notebook was already run against that URL once to pre-populate
`checkpoints/resnet50_5x_224x224_scene_tag_bdd100k.pth` (94,411,924 bytes, matching the
original host's `Content-Length` from before it went down), so `CKPT_PATH.exists()` below
should already be `True` and this cell should just report "Using cached checkpoint."

The checkpoint's internal structure was statically verified (via `zipfile` + a scan of the
pickle byte stream — *not* by unpickling/executing it, which would be unsafe for a file
sourced from a third-party mirror) to contain exactly the `backbone.conv1` /
`backbone.bn1` / `backbone.layer1..4.*.{conv,bn}{1,2,3}` / `backbone.layer*.0.downsample.{0,1}`
/ `head.fc` key layout the loader in the next cell expects — i.e. a stock, non-deep-stem
ResNet-50, one-for-one compatible with `torchvision.models.resnet50`'s parameter names.
No official MD5 could be cross-checked (the `.md5` file itself was never archived), so this
is a strong structural check, not a cryptographic provenance guarantee.

In [3]:
CKPT_URL          = 'https://dl.cv.ethz.ch/bdd100k/tagging/scene/models/resnet50_5x_224x224_scene_tag_bdd100k.pth'
MD5_URL           = 'https://dl.cv.ethz.ch/bdd100k/tagging/scene/models/resnet50_5x_224x224_scene_tag_bdd100k.md5'
# Fallback: dl.cv.ethz.ch has been down since ~March 2026 (SysCV/bdd100k-models#28). This is
# the Internet Archive's crawl of the same file from 2025-01-29, before the host died.
CKPT_URL_FALLBACK = ('http://web.archive.org/web/20250129052906if_/'
                      'https://dl.cv.ethz.ch/bdd100k/tagging/scene/models/'
                      'resnet50_5x_224x224_scene_tag_bdd100k.pth')
CKPT_PATH = CKPT_DIR / 'resnet50_5x_224x224_scene_tag_bdd100k.pth'


def _download(url, dest):
    import urllib.request
    print(f'Downloading {url} -> {dest}')
    urllib.request.urlretrieve(url, dest)


def _extract_md5(text):
    m = re.search(r'\b[a-fA-F0-9]{32}\b', text)
    if not m:
        raise ValueError(f'No MD5-looking token found in: {text!r}')
    return m.group(0).lower()


def _md5_of(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


if not CKPT_PATH.exists():
    try:
        _download(CKPT_URL, CKPT_PATH)
    except Exception as e:
        print(f'Primary host failed ({e}); falling back to the Wayback Machine mirror.')
        _download(CKPT_URL_FALLBACK, CKPT_PATH)
else:
    print(f'Using cached checkpoint: {CKPT_PATH}  ({CKPT_PATH.stat().st_size:,} bytes)')

try:
    import urllib.request
    with urllib.request.urlopen(MD5_URL) as resp:
        expected_md5 = _extract_md5(resp.read().decode())
    actual_md5 = _md5_of(CKPT_PATH)
    if expected_md5 == actual_md5:
        print(f'MD5 OK: {actual_md5}')
    else:
        print(f'MD5 MISMATCH -- expected {expected_md5}, got {actual_md5}. '
              f'Delete {CKPT_PATH} and re-run this cell.')
except Exception as e:
    # Expected right now: the primary host (and its .md5 file) is down, so this just
    # confirms the checkpoint's own size instead.
    print(f'Could not verify official MD5 ({e}).')
    print(f'Local file: {_md5_of(CKPT_PATH)}  ({CKPT_PATH.stat().st_size:,} bytes; '
          f'expected 94,411,924 if downloaded from the Wayback Machine fallback above)')


Using cached checkpoint: c:\Users\hi2ni\281-s2-group2-final\bdd100k_backbone_embeddings\checkpoints\resnet50_5x_224x224_scene_tag_bdd100k.pth  (94,411,924 bytes)
Could not verify official MD5 (<urlopen error [Errno 11001] getaddrinfo failed>).
Local file: cf75e1f6d54aec1988c4bcd5bad74882  (94,411,924 bytes; expected 94,411,924 if downloaded from the Wayback Machine fallback above)


## 2. Build the backbone

The config (`resnet50_5x_224x224_scene_tag_bdd100k.py` -> `_base_/models/resnet50.py`)
specifies a plain `mmcls` `ResNet` backbone: `depth=50`, `style="pytorch"`
(no deep-stem, no avg-down), `GlobalAveragePooling` neck, `LinearClsHead(2048 -> 7)`.
`mmcls` intentionally keeps this parameter layout compatible with
`torchvision.models.resnet50`, so instead of adding `mmcv-full`/`mmcls` as new project
dependencies, this loads the checkpoint's `backbone.*` weights directly into a
`torchvision` ResNet-50 and drops `fc` -- leaving the 2048-d average-pool output as the
penultimate-layer embedding.

**Verify before trusting the output:** the printed missing/unexpected key report should
show *only* `fc.*` as missing (the classification head we're not using) and nothing
missing under `conv1` / `bn1` / `layer1..4`. If backbone keys are missing too, this
checkpoint doesn't use the plain torchvision-compatible layout assumed here and the
loader needs a real key-remapping table instead of a prefix strip.

This was already checked structurally offline (static pickle-key scan, see the note in the
cell above) and it matches. What has **not** been run is the actual `torch.load` +
`load_state_dict` call below — the machine this notebook was authored on has a broken local
`torch` install (`c10.dll` fails to load; a CPU-only `torch==2.13.0+cpu` wheel installed
cleanly via pip but crashes on import with `WinError 1114`, unrelated to this project's
`environment.yml` env). Run this cell in the project's actual `281-s2-group2` conda env,
where `torch`/`torchvision` are already known-working project dependencies, and confirm the
printed key report before trusting `bdd_model`'s output.

In [4]:
import torch
import torchvision
from torchvision import transforms

def load_bdd100k_resnet50_scene(ckpt_path):
    model = torchvision.models.resnet50(weights=None)

    ckpt = torch.load(ckpt_path, map_location='cpu')
    state_dict = ckpt.get('state_dict', ckpt)

    backbone_sd = {
        k[len('backbone.'):]: v
        for k, v in state_dict.items()
        if k.startswith('backbone.')
    }
    if not backbone_sd:
        raise RuntimeError(
            "No 'backbone.*' keys found in checkpoint state_dict -- top-level key layout "
            f"differs from the expected mmcls format. Keys seen: {list(state_dict.keys())[:10]}"
        )

    missing, unexpected = model.load_state_dict(backbone_sd, strict=False)
    print(f'load_state_dict: {len(missing)} missing, {len(unexpected)} unexpected')
    if missing:
        print(f'  missing:    {missing}')
    if unexpected:
        print(f'  unexpected: {unexpected}')
    non_fc_missing = [k for k in missing if not k.startswith('fc.')]
    if non_fc_missing:
        raise RuntimeError(
            'Backbone keys missing beyond the expected fc.* -- the torchvision-compatible '
            f'key-layout assumption does not hold for this checkpoint: {non_fc_missing}'
        )

    model.fc = torch.nn.Identity()  # strip the classification head -> penultimate 2048-d embedding
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


bdd_model = load_bdd100k_resnet50_scene(CKPT_PATH)

# Preprocessing matches the model zoo's test_pipeline for this config: resize short side to
# 256, center-crop 224, normalize with ImageNet stats.
bdd_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Backbone ready. Embedding dim: 2048')


load_state_dict: 2 missing, 0 unexpected
  missing:    ['fc.weight', 'fc.bias']
Backbone ready. Embedding dim: 2048


## 3. Extract embeddings over the CENTER_SEG crop

Same loop shape as the DINOv2 extraction cell in `feature_pipeline_v3_3class.ipynb`: one embedding per sequence, from the target frame's `CENTER_SEG` crop.

In [ ]:
BDD_PATH        = FEATURES_DIR / 'bdd100k_resnet50_scene_3class.npy'
BDD_SEQIDS_PATH = FEATURES_DIR / 'bdd100k_resnet50_scene_3class_seq_ids.npy'
RECOMPUTE_BDD   = not BDD_PATH.exists()

if RECOMPUTE_BDD:
    print(f'Extracting BDD100K ResNet-50 scene embeddings for {len(seq_ids_3c)} sequences...')
    t0 = time.time()
    bdd_list = []

    with torch.no_grad():
        for i, sid in enumerate(seq_ids_3c):
            target_fname = manifest[sid]['context_fnames'][-1]
            img = load_frame(target_fname)

            # CENTER_SEG crop -- consistent with all other features in the pipeline
            x0, x1 = CENTER_SEG
            crop = img[:, x0:x1, :]  # (H, 128, 3)

            pil_img = Image.fromarray(crop)
            inp = bdd_transform(pil_img).unsqueeze(0)  # (1, 3, 224, 224)
            embedding = bdd_model(inp).squeeze(0).numpy()
            bdd_list.append(embedding.astype(np.float32))

            if (i + 1) % 200 == 0:
                elapsed = time.time() - t0
                eta = elapsed / (i + 1) * (len(seq_ids_3c) - (i + 1))
                print(f'  {i+1}/{len(seq_ids_3c)}  ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)')

    bdd_3c = np.array(bdd_list, dtype=np.float32)
    np.save(BDD_PATH, bdd_3c)
    np.save(BDD_SEQIDS_PATH, seq_ids_3c)
    print(f'Done in {time.time()-t0:.0f}s.  Shape: {bdd_3c.shape}')
else:
    bdd_3c = np.load(BDD_PATH)
    print(f'Loaded cached bdd100k_resnet50_scene_3class.npy: {bdd_3c.shape}')

print(f'BDD100K ResNet-50 scene embedding dim: {bdd_3c.shape[1]}')


Extracting BDD100K ResNet-50 scene embeddings for 1604 sequences...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\hi2ni\\281-s2-group2-final\\data\\processed\\waymo_e2e\\training\\003b62820d0e9345eb025de35b046999_10.npz'

## 4. Benchmark

Same SVM/RF setup as the DINOv2 benchmark cell in `feature_pipeline_v3_3class.ipynb`, so results are directly comparable.

In [ ]:
bdd_sets = {
    'BDD-ResNet50':          bdd_3c,
    'BDD-ResNet50+HOG':      np.concatenate([bdd_3c, hog_3c], axis=1),
    'BDD-ResNet50+YOLO':     np.concatenate([bdd_3c, yolo_3c], axis=1),
}
if road_v2_3c is not None:
    bdd_sets['BDD-ResNet50+Road'] = np.concatenate([bdd_3c, road_v2_3c], axis=1)
if dino_3c is not None:
    bdd_sets['BDD-ResNet50+DINO'] = np.concatenate([bdd_3c, dino_3c], axis=1)
bdd_sets['All+BDD-ResNet50'] = np.concatenate(
    [hog_3c, hsv_3c, yolo_3c] + ([road_v2_3c] if road_v2_3c is not None else []) + [bdd_3c],
    axis=1,
)

print(f'Majority-class baseline: {majority_baseline_3c:.3f}')
print('Reference -- DINOv2 alone (feature_pipeline_v3_3class.ipynb): macro-F1 ~= 0.565\n')

bdd_rows = []
for name, feats in bdd_sets.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        feats, labels_3c, test_size=0.25,
        random_state=RANDOM_STATE, stratify=labels_3c)
    sc = StandardScaler()
    X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)

    svm = SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_STATE)
    svm.fit(X_tr_s, y_tr); svm_pred = svm.predict(X_te_s)

    rf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_tr, y_tr); rf_pred = rf.predict(X_te)

    for mname, pred in [('SVM', svm_pred), ('RF', rf_pred)]:
        bdd_rows.append({
            'feature_set': name, 'model': mname,
            'accuracy': accuracy_score(y_te, pred),
            'macro_f1': f1_score(y_te, pred, labels=CLASSES, average='macro', zero_division=0),
        })

    _svm_f1 = f1_score(y_te, svm_pred, labels=CLASSES, average='macro', zero_division=0)
    _rf_f1  = f1_score(y_te, rf_pred,  labels=CLASSES, average='macro', zero_division=0)
    print(f'{name:25s}  SVM f1={_svm_f1:.3f}  |  RF f1={_rf_f1:.3f}')

bdd_df   = pd.DataFrame(bdd_rows)
best_bdd = bdd_df.loc[bdd_df['macro_f1'].idxmax()]
print(f'\nBest BDD-ResNet50 result: {best_bdd["feature_set"]} / {best_bdd["model"]}  '
      f'macro_f1={best_bdd["macro_f1"]:.3f}')
print(f'Delta vs DINOv2 alone (0.565): {best_bdd["macro_f1"] - 0.565:+.3f}')

bdd_df.to_csv(FEATURES_DIR / 'benchmark_bdd100k_resnet50_scene_3class.csv', index=False)
print('Saved benchmark_bdd100k_resnet50_scene_3class.csv')
